# Example: Compare American and European Put Premiums on One Lattice
In this example, we reproduce the two-step put calculation of Hull Examples 13.4 and 13.5 (9th edition, 2015). We price American and European exercise styles on the same binomial lattice, then inspect the node where the American holder's ability to exercise early creates a higher root premium. The rounded factors in this textbook tree are not CRR factors; the backward-induction comparison is the same one used on a CRR tree.

> __Learning Objectives:__
> 
> By the end of this example, you should be able to:
> * __Build a binomial price lattice__ using specified up/down moves and a risk-neutral probability.
> * __Price both exercise styles__ by applying European and American backward-induction rules under identical inputs.
> * __Locate the early-exercise premium__ by identifying the node where immediate exercise beats continuation for the American holder.

Let's get started!
___

## Setup, Data, and Prerequisites
First, we set up the computational environment by including the `Include.jl` file and loading any needed resources.

> __Include:__ The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/). 

Let's set up our code environment:

In [1]:
include(joinpath(@__DIR__, "Include.jl")); # include the Include.jl file

  Activating project at `~/Desktop/julia_work/CHEME-5660-CourseRepository-Fall-2026`


For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/) and the [CHEME 5660 Quantitative Finance Package documentation](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/).

### Constants
In this section, we set some constants. See the comment next to the constant for a description of the constant, its permissible values, etc.

The lecture writes the risk-neutral up probability $q$ and the continuously compounded risk-free rate $g_y$; the code below uses the identifiers `q` and `gᵧ` to match. Each step here is one year, so $\Delta t=1$ yr and the one-step accumulation factor is $e^{g_y}$.

In [2]:
𝒟(g,t) = exp(g*t); # continuous accumulation factor; its inverse discounts
Sₒ = 50.0; # initial share price (USD/share)
h = 2; # levels of the tree starting from zero
T = 2.0; # years until expiration
Δt = T/h; # length of one lattice step: 1.0 yr
u = 1.2; # magnitude of an up move
d = 0.8; # magnitude of a down move
gᵧ = 0.05; # continuously compounded risk-free rate (1/yr)
q = (exp(gᵧ*Δt)-d)/(u-d); # risk-neutral probability of an up move
K = 52; # strike price (USD/share)

___

## Task 1: Populate the Example Lattice From Hull
In this task, we'll construct a lattice model for a put contract. Let's start by calculating the hypothetical share prices of a stock as reproduced from Hull Examples 13.4 and 13.5. 

> __Parameters:__ Consider a put with strike $K=52$ USD/share on a stock at $S_0=50$ USD/share. There are two one-year steps; in each period the stock moves by $u=1.2$ or $d=0.8$. With a continuously compounded risk-free rate $g_y=5\%$, the risk-neutral up probability is $q=(e^{0.05}-0.8)/(1.2-0.8)\approx0.6282$. Because $ud=0.96\neq1$, this is a generic binomial tree rather than a CRR calibration.

Calculating the future share price with a binomial lattice requires setting the model's parameters, including the initial share price (as a `Float64`), the number of time steps to simulate into the future (the number levels of the tree starting from zero) as an `Int64`, the `up` and `down` move magnitudes (as `Float64` values), and the probability of an up move (as a `Float64`). The package names its up-probability keyword `p`; we pass our `q` to it.

Once these values are set, we use [the `build(…)` function](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/equity/#VLQuantitativeFinancePackage.build-Tuple{Type{MyBinomialEquityPriceTree},%20NamedTuple}) to create an empty lattice model [of type `MyBinomialEquityPriceTree`](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/equity/#VLQuantitativeFinancePackage.MyBinomialEquityPriceTree), which is then passed [to the `populate(…)` function](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/equity/#VLQuantitativeFinancePackage.populate-Tuple{MyBinomialEquityPriceTree}) using the [Julia piping operator](https://docs.julialang.org/en/v1/manual/functions/#Function-composition-and-piping) `|>`. The `populate(…)` function calculates the prices and probabilities of each node (stored as `MyBiomialLatticeEquityNodeModel` instances) in the tree:

In [3]:
hull_lattice_model = VLQuantitativeFinancePackage.build(MyBinomialEquityPriceTree, (
        u = u, d = d, p = q, μ = gᵧ, T = T)) |> (x-> populate(x, Sₒ = Sₒ, h = h));

What's in the fields of the `hull_lattice_model::MyBinomialEquityPriceTree` instance? Let's start with the `levels` field, which contains an array of arrays. 

The levels field holds the nodes for each level (time step) of the tree.

In [4]:
hull_lattice_model.levels

Dict{Int64, Vector{Int64}} with 3 entries:
  0 => [0]
  2 => [3, 4, 5]
  1 => [1, 2]

__What's in the data field?__ The `data` field is a dictionary that stores all the individual nodes in the lattice, indexed by their unique node IDs. Each node contains the stock price at that specific point in the tree, and the probability of reaching that node based on the up/down moves and their probabilities.

In [5]:
hull_lattice_model.data

Dict{Int64, MyBiomialLatticeEquityNodeModel} with 6 entries:
  0 => MyBiomialLatticeEquityNodeModel(50.0, 1.0, nothing, nothing)
  4 => MyBiomialLatticeEquityNodeModel(48.0, 0.467141, nothing, nothing)
  5 => MyBiomialLatticeEquityNodeModel(32.0, 0.138252, nothing, nothing)
  2 => MyBiomialLatticeEquityNodeModel(40.0, 0.371822, nothing, nothing)
  3 => MyBiomialLatticeEquityNodeModel(72.0, 0.394607, nothing, nothing)
  1 => MyBiomialLatticeEquityNodeModel(60.0, 0.628178, nothing, nothing)

__How about the connectivity field?__ The `connectivity` field is a dictionary that maps each node to an array of its child node IDs. This defines the tree structure, showing how nodes are connected from parents to children.

In [6]:
hull_lattice_model.connectivity

Dict{Int64, Vector{Int64}} with 3 entries:
  0 => [1, 2]
  2 => [4, 5]
  1 => [3, 4]

___

## Task 2: Compare European and American put premiums
Now use the same price lattice to value two contracts with the same strike and terminal payoff. The European recursion always chooses continuation before expiration; the American recursion chooses the larger of immediate exercise and continuation.

Using the exact risk-neutral probability above gives approximately $5.08963$ USD/share for the American put and $4.19265$ USD/share for the European put. Store both reference values:

In [7]:
reference_price = (american = 5.0896324742, european = 4.1926542806); # USD/share

Next, build [a `MyAmericanPutContractModel` contract instance](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/derivatives/#VLQuantitativeFinancePackage.MyAmericanPutContractModel) using [the `build(...)` function](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/derivatives/#VLQuantitativeFinancePackage.build-Tuple{Type{MyAmericanPutContractModel},%20NamedTuple}). This function takes the type of thing we wish to build as the first argument, and the data needed to build the model as a second argument, encoded in a `NamedTuple` object. The contract field is named `DTE`, but the package measures it in years, so we pass `T`. Save this contract model in the `american_put_contract_model::MyAmericanPutContractModel` variable:

In [8]:
american_put_contract_model = VLQuantitativeFinancePackage.build(MyAmericanPutContractModel, (
        K = K, DTE = T, sense = 1));

Use the continuation-only choice for European exercise, then use the default American choice. Evaluating the American premium second leaves the lattice nodes populated with the American decisions for Task 3:

In [9]:
european_choice = (intrinsic, continuation) -> continuation;
european_put_premium = premium(american_put_contract_model, hull_lattice_model; choice = european_choice);
american_put_premium = premium(american_put_contract_model, hull_lattice_model);
(european = european_put_premium, american = american_put_premium,
 early_exercise_premium = american_put_premium - european_put_premium)

(european = 4.1926542806038585, american = 5.089632474198373, early_exercise_premium = 0.8969781935945145)

Check both premiums against direct two-step calculations. We use [the `isapprox(...)` function](https://docs.julialang.org/en/v1/base/math/#Base.isapprox) with a tight tolerance, and also verify the ordering implied by the larger American exercise set:

In [10]:
@assert isapprox(reference_price.european, european_put_premium; atol = 1e-8)
@assert isapprox(reference_price.american, american_put_premium; atol = 1e-8)
@assert american_put_premium ≥ european_put_premium

___

## Task 3: Inspect the Lattice nodes
In this task, we look at how decisions are made at each node in the lattice, i.e., whether to hold or exercise the contract.
The nodes in the `hull_lattice_model,` which can be accessed using the `data` field, hold two numbers for the `put` contract we just priced.

> * __`intrinsic`__ is the immediate value obtained by exercising the option contract at that node, i.e., $h(S)$ from the lecture.
> * __`value`__ is the option value $V$ at that node: what the optimally managed contract is worth there. At the root it is the premium.

Read the field names carefully. The lecture's __extrinsic value__ is the difference $X = V - h$, that is `value - intrinsic`, and it is not stored on the node. We compute it below.

Let's look at the lattice nodes:

In [11]:
hull_lattice_model.data

Dict{Int64, MyBiomialLatticeEquityNodeModel} with 6 entries:
  0 => MyBiomialLatticeEquityNodeModel(50.0, 1.0, 2.0, 5.08963)
  4 => MyBiomialLatticeEquityNodeModel(48.0, 0.467141, 4.0, 4.0)
  5 => MyBiomialLatticeEquityNodeModel(32.0, 0.138252, 20.0, 20.0)
  2 => MyBiomialLatticeEquityNodeModel(40.0, 0.371822, 12.0, 12.0)
  3 => MyBiomialLatticeEquityNodeModel(72.0, 0.394607, 0.0, 0.0)
  1 => MyBiomialLatticeEquityNodeModel(60.0, 0.628178, 0.0, 1.41475)

The option premium is contained in the root node of the tree (index `0` of the data dictionary field). Notice that the root's `intrinsic` is `2`, while its `value` is larger:

In [12]:
hull_lattice_model.data[0]

MyBiomialLatticeEquityNodeModel(50.0, 1.0, 2.0, 5.089632474198373)

The root therefore splits into the two pieces the lecture defines. Let's compute the extrinsic value $X = V - h$ and the early-exercise premium $A$ side by side, so the three quantities are not confused with one another:

In [13]:
let
    V = hull_lattice_model.data[0].value;      # optimally managed American value at the root
    H = hull_lattice_model.data[0].intrinsic;  # immediate exercise value at the root
    X = V - H;                                 # extrinsic value
    A = american_put_premium - european_put_premium; # early-exercise premium

    println("intrinsic value H      = $(round(H, digits=4)) USD/share");
    println("American value    V    = $(round(V, digits=4)) USD/share");
    println("extrinsic value   X=V-H= $(round(X, digits=4)) USD/share");
    println("early-exercise premium A = $(round(A, digits=4)) USD/share");
    @assert isapprox(X, 3.089632474; atol = 1e-8)
end

intrinsic value H      = 2.0 USD/share
American value    V    = 5.0896 USD/share
extrinsic value   X=V-H= 3.0896 USD/share
early-exercise premium A = 0.897 USD/share


### Example node decision
Starting from the leaves of the tree, decisions are made at each node backward through the tree (backward induction) to arrive at the premium value. While this may seem complicated, it reduces to a series of simple decisions: 

> __Should I hold, or should I exercise?__
> * __Hold__: An agent will hold if they believe the expected discounted future payoff exceeds the immediate reward from exercising the contract. The future will be better than today.
> * __Exercise__: An agent will exercise the contract if the immediate payoff is better than the discounted expected future value of the contract. The future will be worse than today.

Inspect the one-year down node, where $S=40$ USD/share and early exercise creates the difference between the two root premiums:

In [14]:
node_to_look_at = 2; # one-year down node: S = 40 USD/share

Now, let's explore the decision made at `node_to_look_at::Int64`. The indexes of the children of node `node_to_look_at::Int64` are contained in the `connectivity::Dict{Int64, Vector{Int64}}` dictionary. Let's get those kid indexes:

In [15]:
parent = hull_lattice_model.data[node_to_look_at]; # me now
kids = hull_lattice_model.connectivity[node_to_look_at]; # future

Now, let's compute whether we should hold or exercise when we are at `node_to_look_at`:

In [16]:
decision, exercise_value, continuation_value = let

    # Hold: if we hold, we get the expected discounted payoff in the future
    expected_future_value = 0.0;
    counter = 1;
    for i ∈ eachindex(kids)
        k = kids[i];
        V = hull_lattice_model.data[k].value; # option value at the future node
        
        w = q;              # the first child is the up move
        if (counter == 2)
            w = (1-q)       # the second child is the down move
        end

        expected_future_value += (w*V);
        counter += 1;
    end
    continuation_value = (1/𝒟(gᵧ,Δt))*expected_future_value;

    # Exercise: exercise now; we get the intrinsic value of the current node
    S = parent.price;
    exercise_value = max(K-S,0.0); # payoff if we exercise now

    # What should I do?
    decision = nothing;
    if (max(exercise_value, continuation_value) == exercise_value)
        decision = :exercise # We make more by exercising the contract
    else
        decision = :hold
    end

    decision, exercise_value, continuation_value
end;

In [17]:
(exercise_value = exercise_value, continuation_value = continuation_value, decision = decision)

(exercise_value = 12.0, continuation_value = 9.463930074037124, decision = :exercise)

___

## Summary
This example used one lattice to isolate the effect of the exercise convention. The European holder had to wait at every interior node; the American holder could exercise when intrinsic value exceeded continuation.

> __Key Takeaways:__
>
> * __The exercise rule is the only pricing difference:__ Both styles share the lattice, the terminal payoffs, and the discounted expectation. The European recursion always continues, while the American recursion takes the larger of intrinsic and continuation value.
> * __The American premium weakly dominates:__ The American holder can imitate the European holder by waiting, so the American value cannot be smaller under identical inputs.
> * __Strictly higher requires a valuable decision:__ Exercise at the down node, where immediate value beats continuation, is what lifts the root premium here by about nine tenths of a dollar per share.

The market-sized CRR examples apply this same comparison with factors calibrated from volatility and the time step.
___

## Disclaimer and Risks

__This content is offered solely for training and informational purposes__. No offer or solicitation to buy or sell securities or derivative products or any investment or trading advice or strategy is made, given, or endorsed by the teaching team.

__Trading involves risk__. Carefully review your financial situation before investing in securities, futures contracts, options, or commodity interests. Past performance, whether actual or indicated by historical tests of strategies, is no guarantee of future performance or success. Trading is generally inappropriate for someone with limited resources, investment or trading experience, or a low-risk tolerance. Only risk capital that is not required for living expenses.

__You are fully responsible for any investment or trading decisions you make__. Such decisions should be based solely on evaluating your financial circumstances, investment or trading objectives, risk tolerance, and liquidity needs.

___